# Bridge Scour (HEC-18)

An invented crossing: a two-span bridge takes a 140-ft-wide stream down to a
95-ft opening, with a single 3-ft round-nose pier in the channel.  We run the
HEC-18 checks from `civilpy.water_resources.scour` — is the bed live or
clear-water, how much contraction scour, how much local pier scour — first
with hand-entered bed data, then straight from a boring log.

## Design flood hydraulics

Values you would normally pull from a HEC-RAS run at the design (Q100) flood:

| quantity | approach section | contracted (bridge) section |
|---|---|---|
| discharge | 4,200 cfs | 4,200 cfs |
| top width | 140 ft | 95 ft |
| flow depth | 9.5 ft | 10.2 ft |
| mean velocity | 3.2 fps | 4.3 fps |

Streambed is a sandy gravel, D50 ≈ 2 mm.

In [1]:
from civilpy.water_resources import scour

y1, v1 = 9.5, 3.2        # approach depth (ft) and velocity (fps)
y0 = 10.2                # existing depth in the contracted section (ft)
d50 = 2.0                # streambed median grain size (mm)

fr1 = scour.froude_number(v1, y1)
vc = scour.critical_velocity(y1, d50)
print(f"approach Froude number Fr1 = {fr1:.3f}")
print(f"critical velocity Vc = {vc:.2f} fps vs approach V = {v1:.1f} fps")
print("live-bed" if v1 > vc else "clear-water", "contraction scour governs")

approach Froude number Fr1 = 0.183
critical velocity Vc = 3.04 fps vs approach V = 3.2 fps
live-bed contraction scour governs


## Contraction scour

The approach flow is faster than the critical velocity for the bed material,
so sediment is already moving — Laursen's **live-bed** equation (HEC-18
Eq. 6.2) applies.  The clear-water form is shown for comparison; it is the
upper bound you'd use if the bed were armored.

In [2]:
ys_live = scour.live_bed_contraction_scour(
    approach_depth_ft=y1,
    approach_discharge_cfs=4200.0,
    contracted_discharge_cfs=4200.0,
    approach_width_ft=140.0,
    contracted_width_ft=95.0,
    existing_depth_ft=y0,
)
ys_clear = scour.clear_water_contraction_scour(
    contracted_discharge_cfs=4200.0,
    contracted_width_ft=95.0,
    d50_mm=d50,
    existing_depth_ft=y0,
)
print(f"live-bed contraction scour    ys = {ys_live:.2f} ft")
print(f"(clear-water form would give  ys = {ys_clear:.2f} ft)")

live-bed contraction scour    ys = 1.98 ft
(clear-water form would give  ys = 2.41 ft)


## Local pier scour — CSU equation

The 3-ft round-nose pier sits essentially in line with the flow.  With the
contracted-section hydraulics, HEC-18 Eq. 7.1 gives the local scour hole
depth at the pier:

In [3]:
a = 3.0                  # pier width (ft)
y_pier, v_pier = 10.2, 4.3

ys_pier = scour.pier_scour_csu(
    approach_velocity_fps=v_pier,
    approach_depth_ft=y_pier,
    pier_width_ft=a,
    k1=1.0,   # round nose (HEC-18 Table 7.1)
    k2=1.0,   # no skew
    k3=1.1,   # plane-bed condition
)
print(f"local pier scour ys = {ys_pier:.2f} ft  ({ys_pier/a:.2f} pier widths)")

local pier scour ys = 5.46 ft  (1.82 pier widths)


## Skew sensitivity

Angle of attack punishes long piers quickly — K2 grows with both skew and
the length-to-width ratio (HEC-18 Table 7.2):

In [4]:
for skew in (0, 10, 20, 30):
    k2 = scour.angle_of_attack_factor(
        pier_length_ft=25.0, pier_width_ft=a, skew_deg=skew
    )
    ys = scour.pier_scour_csu(v_pier, y_pier, a, k2=k2)
    print(f"skew {skew:2d} deg:  K2 = {k2:.2f}  ->  ys = {ys:.2f} ft")

skew  0 deg:  K2 = 1.00  ->  ys = 5.46 ft
skew 10 deg:  K2 = 1.78  ->  ys = 9.72 ft
skew 20 deg:  K2 = 2.38  ->  ys = 12.97 ft
skew 30 deg:  K2 = 2.86  ->  ys = 15.60 ft


## Straight from a boring log

`pier_scour_from_boring` pulls D50 (and D95 for the coarse-bed armoring
factor K4) from the particle-size analysis nearest the streambed in a
`civilpy.geotech` boring, then applies the same CSU equation.  Here we build
a small synthetic boring; in practice this comes from `parse_diggs` or
`read_pdf_log`.

In [5]:
from civilpy.geotech.boring import Borehole, GradingPoint, GradingResult

# gradation at 2 ft below streambed: a coarse sandy gravel
bed_gradation = GradingResult(2.0, (
    GradingPoint(50.0, 100),   # 50 mm: 100% passing
    GradingPoint(19.0, 88),
    GradingPoint(4.75, 60),
    GradingPoint(2.0, 49),     # D50 ~ 2 mm
    GradingPoint(0.425, 18),
    GradingPoint(0.075, 4),
))
boring = Borehole(boring_id="B-101", total_depth_ft=40.0, grading=[bed_gradation])

result = scour.pier_scour_from_boring(
    boring,
    streambed_depth_ft=2.0,
    approach_velocity_fps=v_pier,
    approach_depth_ft=y_pier,
    pier_width_ft=a,
    pier_length_ft=25.0,
    skew_deg=10.0,
    shape="round",
)
print(f"D50 = {result.d50_mm:.1f} mm, D95 = {result.d95_mm:.0f} mm")
print(f"K1={result.k1}  K2={result.k2:.2f}  K3={result.k3}  K4={result.k4:.2f}")
print(f"local pier scour ys = {result.scour_depth_ft:.2f} ft")

D50 = 2.2 mm, D95 = 33 mm
K1=1.0  K2=1.78  K3=1.1  K4=1.00
local pier scour ys = 9.72 ft


The total scour prism at the pier is the contraction scour plus the local
hole — the number the foundation designer carries into the pile/shaft
capacity run (see the *Geotech Foundations from a Boring Log* notebook).

In [6]:
total = ys_live + result.scour_depth_ft
print(f"total scour at the pier = {ys_live:.2f} + {result.scour_depth_ft:.2f} "
      f"= {total:.2f} ft below the existing bed")

total scour at the pier = 1.98 + 9.72 = 11.70 ft below the existing bed
